# E791 $D^+\to\pi^-\pi^+\pi^+$ — coefficient closure test with adaptive normalization

This notebook repeats the single-fit E791 closure test, but uses `AdaptiveDalitzGrid` for all amplitude and PDF normalization integrals.

The fit setup is intentionally unchanged relative to the regular-grid closure test:

1. E791 Fit-2 coefficients define the injected truth;
2. all resonance masses, widths, spins and radii are fixed;
3. $\rho(770)$ is fixed to $1+0i$ as the reference amplitude;
4. all other coefficients float in Cartesian form $(x,y)$;
5. one toy is generated;
6. one randomized starting point is drawn;
7. exactly one fit is performed.

The only conceptual change is the normalization grid. Refinement is driven by the fixed dynamical amplitudes themselves, not by the fitted coefficients. Therefore the grid can be frozen and reused throughout a coefficient-only fit.


In [ ]:
import numpy as np
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt

from dalitzplotfitter import (
    AdaptiveDalitzGrid,
    DecayChannel, DecayModel, Minimizer, NonResonant, Parameter,
    RealImag, Resonance, enable_x64, weighted_resample,
)

enable_x64()


## 1. E791 Fit-2 injected coefficients


In [ ]:
channel = DecayChannel("D+", ("pi-", "pi+", "pi+"))

fit2_polar = {
    "sigma": (1.17, 205.7),
    "rho770": (1.00, 0.0),
    "NR": (0.48, 57.3),
    "f0_980": (0.43, 165.0),
    "f2_1270": (0.76, 57.3),
    "f0_1370": (0.26, 105.4),
    "rho1450": (0.14, 319.1),
}

def polar_to_xy(r, phase_deg):
    phase = np.deg2rad(phase_deg)
    return r*np.cos(phase), r*np.sin(phase)

def internal_xy(name):
    r, phase = fit2_polar[name]
    if name == "NR":
        phase += 180.0
    return polar_to_xy(r, phase)

truth_xy = {name: internal_xy(name) for name in fit2_polar}


## 2. Build the coefficient-only fit model


In [ ]:
truth = {}

def free_coefficient(name):
    x0, y0 = truth_xy[name]
    truth[f"{name}.x"] = float(x0)
    truth[f"{name}.y"] = float(y0)
    return RealImag(
        Parameter.coefficient(f"{name}.x", 0.0, owner=name, bounds=(-2.0, 2.0), step=0.01),
        Parameter.coefficient(f"{name}.y", 0.0, owner=name, bounds=(-2.0, 2.0), step=0.01),
    )

c = {
    "sigma": free_coefficient("sigma"),
    "rho770": RealImag(1.0, 0.0),
    "NR": free_coefficient("NR"),
    "f0_980": free_coefficient("f0_980"),
    "f2_1270": free_coefficient("f2_1270"),
    "f0_1370": free_coefficient("f0_1370"),
    "rho1450": free_coefficient("rho1450"),
}

components = [
    Resonance("sigma", (0,1), c["sigma"], mass=0.4780, width=0.3240, spin=0, resonance_radius=3.0, parent_radius=3.0),
    Resonance("rho770", (0,1), c["rho770"], mass=0.7693, width=0.1502, spin=1, resonance_radius=3.0, parent_radius=3.0),
    Resonance("f0_980", (0,1), c["f0_980"], mass=0.9750, width=0.0440, spin=0, resonance_radius=3.0, parent_radius=3.0),
    Resonance("f2_1270", (0,1), c["f2_1270"], mass=1.2750, width=0.1850, spin=2, resonance_radius=3.0, parent_radius=3.0),
    Resonance("f0_1370", (0,1), c["f0_1370"], mass=1.4340, width=0.1730, spin=0, resonance_radius=3.0, parent_radius=3.0),
    Resonance("rho1450", (0,1), c["rho1450"], mass=1.4650, width=0.3100, spin=1, resonance_radius=3.0, parent_radius=3.0),
    NonResonant(c["NR"]),
]

model = DecayModel(channel, components)
print("number of free parameters =", sum(not p.fixed for p in model.parameters))


## 3. Build the adaptive normalization grid

Each non-constant dynamical component is used as a refinement probe. The probes are evaluated without fitted coefficient values, so they depend only on the fixed dynamics.

This is the key reason the adaptive grid can be constructed once and cached for the entire coefficient-only fit.


In [ ]:
built_components = model.amplitude_model.components

probes = []
probe_names = []
for component in built_components:
    if component.name == "NR":
        continue
    dynamics = component.function
    probes.append(lambda data, dynamics=dynamics: dynamics(data, None))
    probe_names.append(component.name)

adaptive_builder = AdaptiveDalitzGrid(
    channel.parent_mass,
    channel.daughter_masses,
    base_resolution=48,
    max_depth=5,
    tolerance=0.08,
    max_cells=2_000_000,
)

adaptive = adaptive_builder.build(probes)
norm = adaptive.sample

print("probes            :", probe_names)
print(f"base cells         : {48**2:,}")
print(f"adaptive leaf cells: {adaptive.size:,}")
print("max depth reached  :", int(jnp.max(adaptive.depth)))
print("weight min/max     :", float(jnp.min(norm.weights)), float(jnp.max(norm.weights)))
print("sum leaf uv area   :", float(jnp.sum(adaptive.du * adaptive.dv)))


## 4. Visualize where the adaptive grid refined


In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))
sc = ax.scatter(
    np.asarray(norm.s12), np.asarray(norm.s13),
    c=np.asarray(adaptive.depth), s=1.5, rasterized=True,
)
fig.colorbar(sc, ax=ax, label="refinement depth")
ax.set_xlabel(r"$s_{12}$ [GeV$^2$]")
ax.set_ylabel(r"$s_{13}$ [GeV$^2$]")
ax.set_title("Adaptive normalization grid for the E791 model")
plt.show()


## 5. Quadrature sanity check

The leaf cells partition the auxiliary unit square. Therefore `sum(du*dv)` should be one, and integrating a constant should reproduce the physical Dalitz area.


In [ ]:
integral_one = float(jnp.mean(norm.weights * jnp.ones_like(norm.weights)))
print("integral of 1 =", integral_one)
print("finite weights =", bool(jnp.all(jnp.isfinite(norm.weights))))
assert abs(float(jnp.sum(adaptive.du * adaptive.dv)) - 1.0) < 1e-10


## 6. Generate one pseudo-data sample using the adaptive normalization


In [ ]:
N_POOL = 1_000_000
N_DATA = 100_000

pool = model.generate_phase_space(N_POOL, seed=2000)
truth_cache_pool = model.prepare_cache(pool, norm)
truth_intensity, truth_normalization = truth_cache_pool.evaluate(truth)
target_weights = pool.weights * truth_intensity

data = weighted_resample(
    jax.random.key(791), pool, target_weights, N_DATA, replace=True
)

print("toy events          =", data.size)
print("truth normalization =", float(truth_normalization))


In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 6.5))
h = ax.hist2d(np.asarray(data.s12), np.asarray(data.s13), bins=110)
fig.colorbar(h[3], ax=ax, label="events")
ax.set_xlabel(r"$s_{12}$ [GeV$^2$]")
ax.set_ylabel(r"$s_{13}$ [GeV$^2$]")
ax.set_title("E791 Fit-2 pseudo-data")
plt.show()


## 7. Randomize one starting point


In [ ]:
cache = model.prepare_cache(data, norm)

def nll(values):
    intensity, normalization = cache.evaluate(values)
    return -jnp.sum(jnp.log(jnp.clip(intensity, min=1e-300))) + data.size*jnp.log(normalization)

minimizer = Minimizer(nll, model.parameters, verbose=1)
START_SEED = 314159
start_values = minimizer.random_start(seed=START_SEED)

print(f"{'parameter':16s} {'truth':>11s} {'start':>11s}")
for p in model.parameters:
    if not p.fixed:
        print(f"{p.name:16s} {truth[p.name]:11.6f} {start_values[p.name]:11.6f}")

print("NLL(truth) =", float(nll(truth)))
print("NLL(start) =", float(nll(start_values)))


## 8. Perform exactly one fit


In [ ]:
result = minimizer.fit(start_values=start_values, simplex=False)

print("valid          =", bool(result.valid))
print("NLL fit        =", float(result.fval))
print("NLL truth      =", float(nll(truth)))
print("fit-truth NLL  =", float(result.fval - nll(truth)))
print("EDM            =", float(result.fmin.edm))


## 9. Closure table


In [ ]:
print(f"{'parameter':16s} {'truth':>10s} {'start':>10s} {'fit':>10s} {'error':>10s} {'pull':>9s}")
for p in model.parameters:
    if p.fixed:
        continue
    t = float(truth[p.name])
    s = float(start_values[p.name])
    f = float(result.values[p.name])
    e = float(result.errors[p.name])
    pull = (f-t)/e
    print(f"{p.name:16s} {t:10.5f} {s:10.5f} {f:10.5f} {e:10.5f} {pull:9.3f}")


## 10. Truth vs randomized start vs fitted coefficients


In [ ]:
names = [p.name for p in model.parameters if not p.fixed]
x = np.arange(len(names))
truth_arr = np.array([truth[n] for n in names])
start_arr = np.array([start_values[n] for n in names])
fit_arr = np.array([result.values[n] for n in names], dtype=float)
fit_err = np.array([result.errors[n] for n in names], dtype=float)

fig, ax = plt.subplots(figsize=(13, 5.5))
ax.scatter(x, truth_arr, marker="x", s=70, label="truth")
ax.scatter(x, start_arr, marker="o", s=30, label="random start")
ax.errorbar(x, fit_arr, yerr=fit_err, fmt=".", capsize=2, label="fit")
ax.set_xticks(x)
ax.set_xticklabels(names, rotation=60, ha="right")
ax.set_ylabel("Cartesian coefficient")
ax.set_title("Coefficient closure with adaptive normalization")
ax.legend()
fig.tight_layout()
plt.show()


## 11. Projection: start, fit and truth


In [ ]:
projection_cache = model.prepare_cache(pool, norm)

def projection(values, bins):
    intensity, _ = projection_cache.evaluate(values)
    w = np.asarray(pool.weights * intensity)
    h12, _ = np.histogram(np.asarray(pool.s12), bins=bins, weights=w)
    h13, _ = np.histogram(np.asarray(pool.s13), bins=bins, weights=w)
    return h12 + h13

sdata = np.concatenate([np.asarray(data.s12), np.asarray(data.s13)])
bins = np.linspace(sdata.min(), sdata.max(), 110)
centres = 0.5*(bins[:-1] + bins[1:])
hd, _ = np.histogram(sdata, bins=bins)
hs = projection(start_values, bins)
ht = projection(truth, bins)
fit_values = {p.name: float(result.values[p.name]) for p in model.parameters if not p.fixed}
hf = projection(fit_values, bins)
for h in (hs, ht, hf):
    h *= hd.sum()/h.sum()

fig, ax = plt.subplots(figsize=(10, 5.5))
ax.errorbar(centres, hd, yerr=np.sqrt(np.maximum(hd,1)), fmt=".", label="toy")
ax.step(centres, hs, where="mid", label="random start")
ax.step(centres, hf, where="mid", label="fit")
ax.step(centres, ht, where="mid", linestyle="--", label="truth")
ax.set_xlabel(r"$m^2(\pi^-\pi^+)$ [GeV$^2$]")
ax.set_ylabel("entries / bin")
ax.legend()
plt.show()


## Interpretation

This notebook should be compared directly with `02_fit_dynamic_parameters.ipynb`. Both use the same injected model, toy size, random-start seed and one-fit procedure. The important difference is the normalization quadrature:

- notebook 02: fixed equal-area `DalitzGrid`;
- notebook 06: dynamics-aware `AdaptiveDalitzGrid`.

If both close consistently, the next useful test is a direct numerical comparison of fitted coefficients and normalization integrals using the *same toy sample*.
